# 04 · Core ML export

Converts the trained model to `UNetUltra.mlpackage` (iOS 17+) and puts it next to
`calibration.json`, ready for the app's Inference tab.

- input `image`: 512×512 RGB (the centre square of the frame, cropped and resized by the app);
  the 1/255 scaling used in training is built in
- input `meta`: `[1, 2]` float32 = `[log(calibrated focus distance in m), fx of the 512×512 crop / 1000 − 1]`
- outputs `mask_logits` and `depth` (metres)

Run this on the Mac: comparing Core ML with PyTorch needs macOS.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
from unetultra.export import export_coreml
from unetultra.train import load_model

model, checkpoint = load_model(MODELS / "model.pt")
package = MODELS / "UNetUltra.mlpackage"
mlmodel = export_coreml(model, checkpoint["classes"], package)
print("wrote", package, "· classes", checkpoint["classes"])

## Core ML against PyTorch on real samples

In [ ]:
import platform
from unetultra.calibration import LensCalibration
from unetultra.dataset import SessionDataset
from unetultra.export import compare_with_pytorch
from unetultra.session import find_sessions

DEPTH_TOLERANCE_M = 1e-3
LOGIT_TOLERANCE = 1e-2

if platform.system() != "Darwin":
    print("Core ML models only run on macOS: skipping the comparison on", platform.system())
else:
    calibration = LensCalibration.load(MODELS / "calibration.json")
    samples = SessionDataset(find_sessions(DATA), calibration)
    indices = list(range(0, len(samples), max(1, len(samples) // 8)))[:8]
    comparison = compare_with_pytorch(mlmodel, model, samples, indices)
    for row in comparison:
        print(f"{row['sample_id']}: mask logits {row['mask_logits_max_abs_diff']:.2e}, "
              f"depth {row['depth_max_abs_diff_m'] * 1000:.3f} mm, mask pixels changed {row['mask_pixels_changed']}")
    worst_depth = max(row["depth_max_abs_diff_m"] for row in comparison)
    worst_logit = max(row["mask_logits_max_abs_diff"] for row in comparison)
    print(f"\nmaximum absolute difference: depth {worst_depth * 1000:.3f} mm, mask logits {worst_logit:.2e}")
    assert worst_depth < DEPTH_TOLERANCE_M and worst_logit < LOGIT_TOLERANCE, "Core ML and PyTorch disagree"
    print("PASS")

## Hand over to the app

Both files must exist. Copy them into `ios/UNetUltra/UNetUltra/Resources/` (Xcode picks them up
automatically and compiles the model), or AirDrop them to the phone and import them from the
Inference tab. Neither file is committed to git.

In [ ]:
import shutil

calibration_file = MODELS / "calibration.json"
assert package.exists() and calibration_file.exists()
resources = Path("../ios/UNetUltra/UNetUltra/Resources")
resources.mkdir(exist_ok=True)
# Bundled under another name: Xcode generates a class named after the file, and "UNetUltra"
# is already the app module's name.
shutil.copytree(package, resources / "UNetUltraModel.mlpackage", dirs_exist_ok=True)
shutil.copy(calibration_file, resources / calibration_file.name)
print("copied to", resources.resolve())